In [5]:
# ------------------------------------------------------------
# Diabetes Progression Classification (Standalone Compatible)
# ------------------------------------------------------------

import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


def run_diabetes_pipeline():
    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------
    data = load_diabetes()
    X_raw = pd.DataFrame(data.data, columns=data.feature_names)
    y_raw = pd.Series(data.target)

    # Convert to binary classification
    y = (y_raw > y_raw.median()).astype(int)

    # --------------------------------------------------------
    # Train/test split BEFORE scaling
    # --------------------------------------------------------
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42
    )

    # --------------------------------------------------------
    # Scaling (fit ONLY on training data)
    # --------------------------------------------------------
    scaler = StandardScaler()

    X_train = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=X_raw.columns
    )
    X_test = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=X_raw.columns
    )

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model = LogisticRegression(max_iter=300)
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------
    y_pred = model.predict(X_test)

    print("\n=== Diabetes Classification Results ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    # --------------------------------------------------------
    # Interactive Prediction (REAL-WORLD INPUTS)
    # --------------------------------------------------------
    print("\n--- Interactive Diabetes Progression Prediction ---")

    feature_prompts = {
        'age': {'prompt': 'Enter Age (e.g., 19-95 years): ', 'range': (19, 95)},
        'sex': {'prompt': 'Enter Sex (0 for female, 1 for male): ', 'range': (0, 1)},
        'bmi': {'prompt': 'Enter Body Mass Index (BMI, e.g., 18-40 kg/m²): ', 'range': (18, 40)},
        'bp': {'prompt': 'Enter Average Blood Pressure (e.g., 80-180 mmHg): ', 'range': (80, 180)},
        's1': {'prompt': 'Enter Total Cholesterol (TC, e.g., 120-240 mg/dL): ', 'range': (120, 240)},
        's2': {'prompt': 'Enter LDL (low-density lipoproteins, e.g., 70-160 mg/dL): ', 'range': (70, 160)},
        's3': {'prompt': 'Enter HDL (high-density lipoproteins, e.g., 40-70 mg/dL): ', 'range': (40, 70)},
        's4': {'prompt': 'Enter TC/HDL ratio (e.g., 2-6): ', 'range': (2, 6)},
        's5': {'prompt': 'Enter Glucose (GLU, e.g., 70-130 mg/dL): ', 'range': (70, 130)},
        's6': {'prompt': 'Enter Blood Sugar Level (e.g., 80-150 mg/dL): ', 'range': (80, 150)}
    }

    user_input = {}

    for feature in X_raw.columns:
        prompt_info = feature_prompts[feature]

        while True:
            try:
                value = float(input(prompt_info['prompt']))
                min_v, max_v = prompt_info['range']

                if not (min_v <= value <= max_v):
                    print(f"Value must be between {min_v} and {max_v}")
                    continue

                user_input[feature] = value
                break

            except ValueError:
                print("Invalid input. Please enter a numeric value.")

    # --------------------------------------------------------
    # Create DataFrame WITH FEATURE NAMES (fixes warning)
    # --------------------------------------------------------
    user_df_unscaled = pd.DataFrame([user_input])[X_raw.columns]

    user_df_scaled = pd.DataFrame(
        scaler.transform(user_df_unscaled),
        columns=X_raw.columns
    )

    # Predict
    prediction = model.predict(user_df_scaled)[0]

    label = "Higher Progression Risk" if prediction == 1 else "Lower Progression Risk"
    print(f"\nPrediction: {label}")
    print(f"Raw output: {prediction}")

    return model, scaler


# ------------------------------------------------------------
# Standalone execution
# ------------------------------------------------------------
if __name__ == "__main__":
    run_diabetes_pipeline()




=== Diabetes Classification Results ===
Accuracy: 0.7303370786516854
              precision    recall  f1-score   support

           0       0.77      0.73      0.75        49
           1       0.69      0.72      0.71        40

    accuracy                           0.73        89
   macro avg       0.73      0.73      0.73        89
weighted avg       0.73      0.73      0.73        89


--- Interactive Diabetes Progression Prediction ---


Enter Age (e.g., 19-95 years):  20
Enter Sex (0 for female, 1 for male):  0
Enter Body Mass Index (BMI, e.g., 18-40 kg/m²):  18
Enter Average Blood Pressure (e.g., 80-180 mmHg):  80
Enter Total Cholesterol (TC, e.g., 120-240 mg/dL):  120
Enter LDL (low-density lipoproteins, e.g., 70-160 mg/dL):  70
Enter HDL (high-density lipoproteins, e.g., 40-70 mg/dL):  40
Enter TC/HDL ratio (e.g., 2-6):  2
Enter Glucose (GLU, e.g., 70-130 mg/dL):  70
Enter Blood Sugar Level (e.g., 80-150 mg/dL):  80



Prediction: Higher Progression Risk
Raw output: 1


In [6]:
import joblib
import os

PROJECT_ROOT = os.path.abspath("..")   # notebook inside /notebooks
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "diabetes_model.pkl")

joblib.dump((model, scaler), model_path)

print(f"✅ Model and scaler saved to {model_path}")


✅ Model and scaler saved to C:\Users\charo\ai-portfolio\models\diabetes_model.pkl
